<a href="https://colab.research.google.com/github/Suman18-bit/NLP_CHALANGES/blob/main/Custom_model_training_with_spacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
import spacy

In [44]:
import pandas as pd
from spacy.tokens import DocBin

In [45]:
import ast

In [46]:
df=pd.read_csv("01_03_data_annotation_for_named_entity_recognition.csv")

In [47]:
df.head()

,text,annotations
0,The company was founded in December 2002 by R...,(' The company was founded in December 2002 by...
1,"In June 2008, Sequoia Capital, Greylock Partne...","('In June 2008, Sequoia Capital, Greylock Part..."
2,LinkedIn office building at 222 Second Street ...,('LinkedIn office building at 222 Second Stree...
3,LinkedIn office in Toronto inside the Toronto ...,('LinkedIn office in Toronto inside the Toront...
4,LinkedIn filed for an initial public offering ...,('LinkedIn filed for an initial public offerin...


In [48]:
df['annotations']=df['annotations'].apply(ast.literal_eval)

In [49]:
df.head()

,text,annotations
0,The company was founded in December 2002 by R...,( The company was founded in December 2002 by ...
1,"In June 2008, Sequoia Capital, Greylock Partne...","(In June 2008, Sequoia Capital, Greylock Partn..."
2,LinkedIn office building at 222 Second Street ...,(LinkedIn office building at 222 Second Street...
3,LinkedIn office in Toronto inside the Toronto ...,(LinkedIn office in Toronto inside the Toronto...
4,LinkedIn filed for an initial public offering ...,(LinkedIn filed for an initial public offering...


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   text         15 non-null     object
 1   annotations  15 non-null     object
dtypes: object(2)
memory usage: 372.0+ bytes


In [51]:
N=100

In [52]:
df=pd.concat([df]*N,ignore_index=True)

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   text         1500 non-null   object
 1   annotations  1500 non-null   object
dtypes: object(2)
memory usage: 23.6+ KB


In [54]:
train_data = df['annotations'][:1200].tolist()
test_data = df['annotations'][1200:].tolist()

In [55]:
nlp = spacy.blank("en")

In [56]:
db = DocBin()

In [57]:
for text, annot in train_data:
  doc=nlp.make_doc(text)
  ent = []
  for star,end,label in annot['entities']:
    span = doc.char_span(star,end,label=label)
    if span is not None:
      ent.append(span)
  doc.ents = ent
  db.add(doc)
db.to_disk("./train.spacy")

In [58]:
nlp2 = spacy.blank("en")
db2 = DocBin()

In [59]:
for text, annot in test_data:
  doc=nlp.make_doc(text)
  ent = []
  for star,end,label in annot['entities']:
    span = doc.char_span(star,end,label=label)
    if span is not None:
      ent.append(span)
  doc.ents = ent
  db.add(doc)
db.to_disk("./test.spacy")

In [65]:
!python -m spacy init config config.cfg --lang en --pipeline ner --force
!python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./test.spacy

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy

============================ Data file validation ============================
✔ Pipeline can be initialized with data
✔ Corpus is loadable

=============================== Training stats ===============================
Language: en
Training pipeline: tok2vec, ner
1200 training docs
1500 evaluation docs
⚠ 15 training examples also in evaluation data
⚠ Low number of examples to train a new pipeline (1200)

============================== Vocab & Vectors ===========

In [68]:
!python -m spacy init fill-config config.cfg base_config.cfg

✔ Auto-filled config with all values
✔ Saved config
base_config.cfg
You can now add your data and train your pipeline:
python -m spacy train base_config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [69]:
!python -m spacy train /content/config.cfg --output ./ner_output

✔ Created output directory: ner_output
ℹ Saving to output directory: ner_output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     65.77    0.00    0.00    0.00    0.00
  0     200       7120.11   2774.43  100.00  100.00  100.00    1.00
  0     400         60.64    112.12  100.00  100.00  100.00    1.00
  0     600          2.23      1.50  100.00  100.00  100.00    1.00
  1     800          0.00      0.00  100.00  100.00  100.00    1.00
  1    1000          1.15      0.56  100.00  100.00  100.00    1.00
  1    1200         12.56      5.84  100.00  100.00  100.00    1.00
  2    1400        187.69     83.27  100.00  100.00  100

In [70]:
test_nlp = spacy.load("/content/ner_output/model-best")

In [72]:
test_nlp.get_pipe("ner").labels

('company', 'date', 'money', 'person', 'place')

In [89]:
test = """In June 2008, Sequoia Capital, Greylock Partners, and other venture capital firms purchased a 5% stake in the company for $53 million, giving the company a post-money valuation of approximately $1 billion. In November 2009, LinkedIn opened its office in Mumbai and soon thereafter in Sydney, as it started its Asia-Pacific team expansion. In 2010 LinkedIn opened an International Headquarters in Dublin, Ireland,received a $20 million investment from Tiger Global Management LLC at a valuation of approximately $2 billion,announced its first acquisition, Mspoke,and improved its 1% premium subscription ratio. In October of that year, Silicon Valley Insider ranked the company No. 10 on its Top 100 List of most valuable startups. By December, the company was valued at $1.575 billion in private markets. LinkedIn started its India operations in 2009 and a major part of the first year was dedicated to understanding professionals in India and educating members to leverage LinkedIn for career development """

In [90]:
doc = test_nlp(test)

In [91]:
doc

In June 2008, Sequoia Capital, Greylock Partners, and other venture capital firms purchased a 5% stake in the company for $53 million, giving the company a post-money valuation of approximately $1 billion. In November 2009, LinkedIn opened its office in Mumbai and soon thereafter in Sydney, as it started its Asia-Pacific team expansion. In 2010 LinkedIn opened an International Headquarters in Dublin, Ireland,received a $20 million investment from Tiger Global Management LLC at a valuation of approximately $2 billion,announced its first acquisition, Mspoke,and improved its 1% premium subscription ratio. In October of that year, Silicon Valley Insider ranked the company No. 10 on its Top 100 List of most valuable startups. By December, the company was valued at $1.575 billion in private markets. LinkedIn started its India operations in 2009 and a major part of the first year was dedicated to understanding professionals in India and educating members to leverage LinkedIn for career develo

In [92]:
spacy.displacy.render(doc,style="ent",jupyter=True)